In [1]:
import requests
from typing import List, Dict, Any, Optional

def search_products_by_ingredients(ingredients: List[str], max_results: int = 10, query: str = "*", status: Optional[int] = None) -> List[Dict[str, Any]]:
    """
    Search the NIH Dietary Supplement Label Database (DSLD) for products 
    that contain all of the specified ingredients.
    
    Args:
        ingredients (List[str]): A list of ingredient names (e.g. ['Vitamin C', 'Zinc']).
        max_results (int): The max number of products to return (default 10).
        query (str): Optional search term if you want to narrow down further (default '*').
        status (int): Market status filter. 1 = on market, 0 = off market, 2 = all.
        
    Returns:
        List[Dict[str, Any]]: A list of dictionaries representing the products.
    """
    base_url = "https://api.ods.od.nih.gov/dsld/v9/search-filter"
    
    # The API expects ingredients to be a comma-separated string
    # When multiple ingredients are given, it treats them as a logical AND (requires ALL of them)
    ingredient_param = ",".join(ingredients)
    
    params = {
        "q": query,
        "ingredient_name": ingredient_param,
        "size": max_results
    }
    if status is not None:
        params["status"] = status
        
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    
    data = response.json()
    
    # Extract hit metadata and source product info
    products = []
    for hit in data.get("hits", []):
        product_info = hit.get("_source", {})
        # Attach the DSLD ID so we can look up detailed label info later if needed
        product_info["dsld_id"] = hit.get("_id")
        products.append(product_info)
        
    return products

In [2]:
# Example usage: Find products containing multiple ingredients
ingredients_to_search = ["Melatonin", "L-Theanine"]
results = search_products_by_ingredients(ingredients_to_search, max_results=3, status=1)

for idx, product in enumerate(results, 1):
    brand = product.get("brandName", "Unknown Brand")
    name = product.get("fullName", "Unknown Product Name")
    print(f"{idx}. {brand} - {name} (ID: {product.get('dsld_id')})")

1. HoltraCeuticals - Sleep Tight (ID: 270840)
2. Nature Made - Melatonin + 200 mg L-Theanine (ID: 271095)
3. youtheory - Sleep Nighttime Powder (ID: 272057)
